# Data Degradation Demo (VOC2012)
用于生成并可视化数据退化后的样本结果，优先复用 `src/degradations.py`。

In [ ]:
import sys, json, random
from pathlib import Path
import numpy as np
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

try:
    import cv2
except Exception:
    cv2 = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path(r"e:/gd/ECE271B/Final proj/degradation_aware_seg_project")
sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / 'data' / 'VOCdevkit' / 'VOC2012' / 'JPEGImages'
OUT_DIR = PROJECT_ROOT / 'artifacts' / 'degradation_demo'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)
print(DATA_ROOT)
print(OUT_DIR)

In [ ]:
deg_module = None
project_degrade_callable = None
try:
    import src.degradations as deg_module
except Exception as e:
    print('cannot import src.degradations:', repr(e))

if deg_module is not None:
    for name in ['apply_degradation','apply_degradations','degrade_image','degrade','random_degradation','build_degradation_pipeline']:
        if hasattr(deg_module, name) and callable(getattr(deg_module, name)):
            project_degrade_callable = getattr(deg_module, name)
            print('use:', name)
            break

In [ ]:
def to_uint8_np(img):
    arr = np.array(img)
    return np.clip(arr, 0, 255).astype(np.uint8)

def from_np(arr):
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def add_gaussian_noise(img, sigma=18):
    arr = to_uint8_np(img).astype(np.float32)
    return from_np(arr + np.random.normal(0, sigma, arr.shape))

def add_motion_blur(img, ksize=13):
    if cv2 is None:
        return img.filter(ImageFilter.GaussianBlur(radius=2))
    arr = to_uint8_np(img)
    kernel = np.zeros((ksize, ksize), dtype=np.float32)
    kernel[ksize // 2, :] = 1.0
    kernel /= kernel.sum()
    return from_np(cv2.filter2D(arr, -1, kernel))

def add_defocus_blur(img, radius=2):
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def add_jpeg_compression(img, quality=25):
    import io
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

def reduce_resolution(img, scale=0.5):
    w, h = img.size
    img2 = img.resize((max(1, int(w*scale)), max(1, int(h*scale))), Image.BILINEAR)
    return img2.resize((w, h), Image.BILINEAR)

def adjust_low_light(img, factor=0.45):
    arr = to_uint8_np(img).astype(np.float32)
    return from_np(arr * factor)

def fallback_degradations(img):
    return {
        'gaussian_noise': add_gaussian_noise(img),
        'motion_blur': add_motion_blur(img),
        'defocus_blur': add_defocus_blur(img),
        'jpeg_25': add_jpeg_compression(img),
        'downsample_x0.5': reduce_resolution(img),
        'low_light': adjust_low_light(img),
    }

def normalize_project_output(out):
    if out is None:
        return None
    if isinstance(out, dict):
        rst = {}
        for k, v in out.items():
            if isinstance(v, Image.Image):
                rst[str(k)] = v.convert('RGB')
            elif isinstance(v, np.ndarray):
                rst[str(k)] = from_np(v)
        return rst if rst else None
    if isinstance(out, Image.Image):
        return {'project_degraded': out.convert('RGB')}
    if isinstance(out, np.ndarray):
        return {'project_degraded': from_np(out)}
    return None

def run_project_degradation_if_possible(img):
    if project_degrade_callable is None:
        return None
    tries = [
        lambda: project_degrade_callable(img),
        lambda: project_degrade_callable(np.array(img)),
        lambda: project_degrade_callable(image=img),
        lambda: project_degrade_callable(image=np.array(img)),
    ]
    for fn in tries:
        try:
            return fn()
        except Exception:
            pass
    return None

In [ ]:
imgs = sorted(DATA_ROOT.glob('*.jpg'))
assert imgs, f'No jpg in {DATA_ROOT}'
random.seed(271)
sample_paths = random.sample(imgs, k=min(3, len(imgs)))

def make_grid(img_path):
    img = Image.open(img_path).convert('RGB')
    po = normalize_project_output(run_project_degradation_if_possible(img))
    if po:
        degs = po
        source = 'project'
    else:
        degs = fallback_degradations(img)
        source = 'fallback'
    items = [('original', img)] + list(degs.items())
    cols = 3
    rows = int(np.ceil(len(items)/cols))
    fig = plt.figure(figsize=(5*cols, 4*rows))
    for i, (name, im) in enumerate(items, 1):
        ax = fig.add_subplot(rows, cols, i)
        ax.imshow(im)
        ax.set_title(name)
        ax.axis('off')
    fig.suptitle(f'{img_path.name} | source={source}')
    fig.tight_layout()
    out = OUT_DIR / f'{img_path.stem}_degradation_grid.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return {'image': img_path.name, 'output': str(out), 'source': source, 'degradations': list(degs.keys())}

summary = [make_grid(p) for p in sample_paths]
with open(OUT_DIR / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
summary